# s14 — Prompt-Cache Stats

**What this teaches:** how to attach the `internal/cache` plugin to a runner so the agent reports its prompt-cache hit rate after a run. The cache itself lives at the provider (Anthropic / OpenAI / Gemini) — this plugin only *observes* the usage numbers each model response returns and aggregates them into a session-wide summary.

**Concept anchor:** providers will return cached input tokens for free (or near-free) when consecutive turns share a long prefix. The plugin computes `cached / (cached + uncached)` per request and exposes it via `stats.Summary()`. The metric you watch is **cache hit rate** — a healthy long-running agent should trend high.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider whose API returns cache-usage fields. Anthropic returns `cache_read_input_tokens`/`cache_creation_input_tokens` natively; OpenAI exposes `prompt_tokens_details.cached_tokens`. Configure as usual:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```

## 1. Imports

We add `internal/cache` to the imports you already know from [s01_loop](../s01_loop/s01_loop.ipynb).

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/cache"
)

## 2. Helper

Same panic-flavoured `must` we use throughout the notebooks.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the cache stats plugin

`cache.Plugin(name)` returns two things: a `*Stats` handle you query at the end, and an ADK plugin you hand to the runner. The plugin hooks into `after_model` to read usage fields out of every response.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
stats, plug, err := cache.Plugin("cache")
must(err)
fmt.Println("plugin wired:", plug)

## 4. Build agent and runner

The plugin is the third (variadic) argument to `Runner`. Everything else is identical to the bare-loop example.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:  "s14_cache",
    Model: llm,
})
must(err)
r, err := agentkit.Runner("s14", a, plug)
must(err)

## 5. Run and print the summary

After the loop terminates, `stats.Summary()` prints a one-line report with totals and the **cache hit rate** as a percentage. The first turn of a session usually has 0 % — the cache has to be written before it can be read.

In [ ]:
prompt := "Repeat: hello."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))
fmt.Println(stats.Summary())

## What to look for

- The `stats.Summary()` line is the only new output vs. [s01_loop](../s01_loop/s01_loop.ipynb). Hit-rate climbs as you reuse the same session ID across turns — pair this notebook with [s16_resume](../s16_resume/s16_resume.ipynb) to see it grow.
- Compression rewrites the prompt prefix and therefore *resets* the cache. The cache plugin and the compression plugin are designed to be read together — see [s20_compress](../s20_compress/s20_compress.ipynb).

## Try it yourself

1. Re-run the final cell several times with the same `prompt`. The first run prints ~0 % hit rate; subsequent runs against the cached prefix should climb.
2. Switch `YOKE_PROVIDER` to one that does not expose cache fields (some `openai_compat` endpoints) and observe that the summary still renders but reports 0 cached tokens.